In [2]:
!pip install boto3

  Using cached boto3-1.43.2-py3-none-any.whl (140 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl (20 kB)
  Using cached s3transfer-0.17.0-py3-none-any.whl (86 kB)
  Using cached botocore-1.43.2-py3-none-any.whl (15.0 MB)
     ---------------------------------------- 0.0/131.6 kB ? eta -:--:--
     --- ------------------------------------ 10.2/131.6 kB ? eta -:--:--
     ----- ------------------------------- 20.5/131.6 kB 222.6 kB/s eta 0:00:01
     -------- ---------------------------- 30.7/131.6 kB 220.2 kB/s eta 0:00:01
     ----------- ------------------------- 41.0/131.6 kB 219.4 kB/s eta 0:00:01
     ----------------- ------------------- 61.4/131.6 kB 273.8 kB/s eta 0:00:01
     ----------------- ------------------- 61.4/131.6 kB 273.8 kB/s eta 0:00:01
     -------------------- ---------------- 71.7/131.6 kB 231.8 kB/s eta 0:00:01
     ---------------------------- ------- 102.4/131.6 kB 295.4 kB/s eta 0:00:01
     ------------------------------------ 131.6/131.6 kB 323.6 kB/s 


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\monaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
!pip install requests

     ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
     ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
     ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
     ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
     ------ --------------------------------- 10.2/64.9 kB ? eta -:--:--
     ------ --------------------------------- 10.2/64.9 kB ? eta -:--:--
     ------ --------------------------------- 10.2/64.9 kB ? eta -:--:--
     ------ --------------------------------- 10.2/64.9 kB ? eta -:--:--
     ------ --------------------------------- 10.2/64.9 kB ? eta -:--:--
     ------ --------------------------------- 10.2/64.9 kB ? eta -:--:--
     ------------------ -------------------- 30.7/64.9 kB 81.9 kB/s eta 0:00:01
     ------------------ -------------------- 30.7/64.9 kB 81.9 kB/s eta 0:00:01
     ------------------------ -------------- 41.0/64.9 kB 89.3 kB/s eta 0:00:01
     -----------------------------


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\monaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import boto3
from credentials import AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN

# Authenticate access to AWS S3
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key= AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name='us-east-1'
)

print(s3.list_buckets())

{'ResponseMetadata': {'RequestId': 'CN8E490SJ8911V0T', 'HostId': 'ai8pNLTUIRL+HtPKjESElFPqWCuQpWExedYN6lN31U/WkyqHD6M6gbWlAb+d4VN5ERyYQrg8MmA=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'ai8pNLTUIRL+HtPKjESElFPqWCuQpWExedYN6lN31U/WkyqHD6M6gbWlAb+d4VN5ERyYQrg8MmA=', 'x-amz-request-id': 'CN8E490SJ8911V0T', 'date': 'Mon, 04 May 2026 16:59:57 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'acharya-cui-sokolenko-mwaa', 'CreationDate': datetime.datetime(2026, 2, 24, 19, 9, 1, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::acharya-cui-sokolenko-mwaa'}, {'Name': 'acharya-de300-wi26', 'CreationDate': datetime.datetime(2026, 2, 17, 6, 21, 19, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::acharya-de300-wi26'}, {'Name': 'adams-agrawal-evensen-mwaa', 'CreationDate': datetime.datetime(2025, 6, 7, 20, 58, 20, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::adams-agrawal-evensen-mwaa'}, {'Name': 'ads-de300win

In [ ]:
import boto3
import requests

# Download the movielens dataset zip, and skips it's already in S3
def download_upload_dataset(bucket_name, s3_key="data/ml-1m.zip"):
    
    try:
        s3.head_object(Bucket=bucket_name, Key=s3_key)
        print(f"Dataset already exists in s3://{bucket_name}/{s3_key}. No need to download.")
        return
    except:
        print("Downloading dataset...")
    
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    response = requests.get(url)
    
    s3.put_object(Bucket=bucket_name, Key=s3_key, Body=response.content)
    print(f"Uploaded to s3://{bucket_name}/{s3_key}")

download_upload_dataset("gomaa-hw2")

Uploaded to s3://gomaa-hw2/data/ml-1m.zip


In [ ]:
# Confirm dataset was uploaded to S3
response = s3.list_objects_v2(Bucket="gomaa-hw2")
for obj in response.get("Contents", []):
    print(obj["Key"])

data/ml-1m.zip


In [ ]:
import zipfile
import io
import pandas as pd

# Load movies released pre or in 1980
def load_movies(s3, bucket_name, s3_key="data/ml-1m.zip"):
    obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
    zip_bytes = io.BytesIO(obj["Body"].read())
    
    with zipfile.ZipFile(zip_bytes) as z:
        with z.open("ml-1m/movies.dat") as f:
            movies = pd.read_csv(f, sep="::", engine="python", 
                                 names=["MovieID", "Title", "Genres"], 
                                 encoding="latin-1")
    
    movies["Year"] = movies["Title"].str.extract(r'\((\d{4})\)').astype(int)
    movies_pre1980 = movies[movies["Year"] <= 1980]
    
    return movies_pre1980

movies_pre1980 = load_movies(s3, "gomaa-hw2")
print(f"Movies released on/before 1980: {len(movies_pre1980)}")
movies_pre1980.head()

Movies released on/before 1980: 887


,MovieID,Title,Genres,Year
109,111,Taxi Driver (1976),Drama|Thriller,1976
152,154,Belle de jour (1967),Drama,1967
197,199,"Umbrellas of Cherbourg, The (Parapluies de Che...",Drama|Musical,1964
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,1977
386,390,Faster Pussycat! Kill! Kill! (1965),Action|Comedy|Drama,1965


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Creates new BERT embeddings and saves to S3 (skips if already there)
def create_embeddings(movies_df, s3, bucket_name, s3_key="embeddings/pre1980_embeddings.npy"):
    try:
        s3.head_object(Bucket=bucket_name, Key=s3_key)
        print("Embeddings already exist in S3. Loading...")
        obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
        embeddings = np.load(io.BytesIO(obj["Body"].read()))
        return embeddings
    except:
        print("Creating embeddings...")

    movies_df["text"] = movies_df["Title"] + ": " + movies_df["Genres"].str.replace("|", " ", regex=False)
    
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(movies_df["text"].tolist(), show_progress_bar=True)
    
    buffer = io.BytesIO()
    np.save(buffer, embeddings)
    buffer.seek(0)
    s3.put_object(Bucket=bucket_name, Key=s3_key, Body=buffer.read())
    print(f"Embeddings saved to s3://{bucket_name}/{s3_key}")
    
    return embeddings

embeddings_pre1980 = create_embeddings(movies_pre1980, s3, "gomaa-hw2")
print(f"Embeddings shape: {embeddings_pre1980.shape}")

C:\Users\monaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating embeddings...


C:\Users\monaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\monaa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 

Embeddings saved to s3://gomaa-hw2/embeddings/pre1980_embeddings.npy
Embeddings shape: (887, 384)


In [17]:
def load_ratings_from_s3(s3, bucket_name, s3_key="data/ml-1m.zip"):
    obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
    zip_bytes = io.BytesIO(obj["Body"].read())
    
    with zipfile.ZipFile(zip_bytes) as z:
        with z.open("ml-1m/ratings.dat") as f:
            ratings = pd.read_csv(f, sep="::", engine="python",
                                  names=["UserID", "MovieID", "Rating", "Timestamp"],
                                  encoding="latin-1")
    return ratings

ratings = load_ratings_from_s3(s3, "gomaa-hw2")
ratings.head()

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Finds random top user
def get_top_user(ratings):
    user_counts = ratings.groupby("UserID").size()
    threshold = user_counts.quantile(0.95)
    top_users = user_counts[user_counts >= threshold].index
    return np.random.choice(top_users)

# Recommends 5 movies for top/cold user and saves to S3
def recommend_movies(user_type, movies_df, embeddings, ratings, s3, bucket_name, top_user_id=None):
    if user_type == "cold":
        popular = ratings.groupby("MovieID").size().reset_index(name="count")
        popular = popular[popular["MovieID"].isin(movies_df["MovieID"])]
        top5 = popular.sort_values("count", ascending=False).head(5)
        recommendations = movies_df[movies_df["MovieID"].isin(top5["MovieID"])][["MovieID", "Title"]]
        user_summary = {"User_Type": "cold", "Last_Interaction_Time": None, "Num_Ratings": 0}

    elif user_type == "top":
        user_ratings = ratings[ratings["UserID"] == top_user_id]
        liked = user_ratings[user_ratings["Rating"] >= 4]["MovieID"]
        liked_indices = movies_df[movies_df["MovieID"].isin(liked)].index
        movies_df_reset = movies_df.reset_index(drop=True)
        liked_indices = movies_df_reset[movies_df_reset["MovieID"].isin(liked)].index
        
        if len(liked_indices) == 0:
            taste_vector = embeddings.mean(axis=0, keepdims=True)
        else:
            taste_vector = embeddings[liked_indices].mean(axis=0, keepdims=True)
        
        sims = cosine_similarity(taste_vector, embeddings)[0]
        top_indices = sims.argsort()[::-1][:5]
        recommendations = movies_df_reset.iloc[top_indices][["MovieID", "Title"]]
        
        last_time = pd.to_datetime(user_ratings["Timestamp"].max(), unit="s")
        user_summary = {"User_Type": "top", "Last_Interaction_Time": str(last_time), 
                        "Num_Ratings": len(user_ratings)}

    result = {**user_summary, "Recommendations": recommendations["Title"].tolist()}
    
    buffer = io.BytesIO(pd.DataFrame([result]).to_json().encode())
    s3.put_object(Bucket=bucket_name, Key=f"recommendations/{user_type}_user.json", Body=buffer.read())
    print(f"Saved to S3: recommendations/{user_type}_user.json")
    
    return result

top_user_id = get_top_user(ratings)
cold_result = recommend_movies("cold", movies_pre1980, embeddings_pre1980, ratings, s3, "gomaa-hw2")
top_result = recommend_movies("top", movies_pre1980, embeddings_pre1980, ratings, s3, "gomaa-hw2", top_user_id)

print(cold_result)
print(top_result)

Saved to S3: recommendations/cold_user.json
Saved to S3: recommendations/top_user.json
{'User_Type': 'cold', 'Last_Interaction_Time': None, 'Num_Ratings': 0, 'Recommendations': ['Star Wars: Episode IV - A New Hope (1977)', 'Godfather, The (1972)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Alien (1979)', 'Airplane! (1980)']}
{'User_Type': 'top', 'Last_Interaction_Time': '2001-02-15 18:18:13', 'Num_Ratings': 649, 'Recommendations': ['Very Natural Thing, A (1974)', 'Windows (1980)', 'Close Encounters of the Third Kind (1977)', 'Mad Max (1979)', 'Apocalypse Now (1979)']}


In [ ]:
# Loads full movie set to S3 without redundancy
def load_all_movies(s3, bucket_name, s3_key="data/ml-1m.zip"):
    obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
    zip_bytes = io.BytesIO(obj["Body"].read())
    
    with zipfile.ZipFile(zip_bytes) as z:
        with z.open("ml-1m/movies.dat") as f:
            movies = pd.read_csv(f, sep="::", engine="python",
                                 names=["MovieID", "Title", "Genres"],
                                 encoding="latin-1")
    
    movies["Year"] = movies["Title"].str.extract(r'\((\d{4})\)').astype(int)
    return movies

all_movies = load_all_movies(s3, "gomaa-hw2")
print(f"Total movies: {len(all_movies)}")

Total movies: 3883


In [ ]:
# Create embeddings for all movies not already saved to S3
all_embeddings = create_embeddings(all_movies, s3, "gomaa-hw2", s3_key="embeddings/all_embeddings.npy")
print(f"Embeddings shape: {all_embeddings.shape}")

Creating embeddings...


Batches: 100%|██████████| 122/122 [00:04<00:00, 24.91it/s]


Embeddings saved to s3://gomaa-hw2/embeddings/all_embeddings.npy
Embeddings shape: (3883, 384)


In [ ]:
# Recommendations for cold/top user given all the movies
colduser_results_all = recommend_movies("cold", all_movies, all_embeddings , ratings, s3, "gomaa-hw2")
topuser_results_all = recommend_movies("top", all_movies, all_embeddings , ratings, s3, "gomaa-hw2", top_user_id)

print(colduser_results_all)
print(topuser_results_all)

Saved to S3: recommendations/cold_user.json
Saved to S3: recommendations/top_user.json
{'User_Type': 'cold', 'Last_Interaction_Time': None, 'Num_Ratings': 0, 'Recommendations': ['Star Wars: Episode IV - A New Hope (1977)', 'Jurassic Park (1993)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'American Beauty (1999)']}
{'User_Type': 'top', 'Last_Interaction_Time': '2001-02-15 18:18:13', 'Num_Ratings': 649, 'Recommendations': ['Blast from the Past (1999)', 'Kids (1995)', 'Back to the Future (1985)', 'Cocoon (1985)', 'Near Dark (1987)']}


In [ ]:
# creates my (self) user profile with ratings for 10 movies, saves recommendations
def create_my_profile(movies_all, embeddings_all, s3, bucket_name):
    my_ratings = [
        {"MovieID": 215,  "Title": "Before Sunrise (1995)",            "Rating": 5},
        {"MovieID": 2572, "Title": "10 Things I Hate About You (1999)","Rating": 3},
        {"MovieID": 586,  "Title": "Home Alone (1990)",                 "Rating": 4},
        {"MovieID": 1,    "Title": "Toy Story (1995)",                  "Rating": 2},
        {"MovieID": 1721, "Title": "Titanic (1997)",                    "Rating": 5},
        {"MovieID": 2025, "Title": "Lolita (1997)",                     "Rating": 1},
        {"MovieID": 165,  "Title": "Die Hard: With a Vengeance (1995)", "Rating": 1},
        {"MovieID": 2273, "Title": "Rush Hour (1998)",                  "Rating": 1},
        {"MovieID": 1704, "Title": "Good Will Hunting (1997)",          "Rating": 5},
        {"MovieID": 1213, "Title": "GoodFellas (1990)",                 "Rating": 1},
    ]
    
    profile_df = pd.DataFrame(my_ratings)
    
    movies_reset = movies_all.reset_index(drop=True)
    liked_ids = profile_df[profile_df["Rating"] >= 4]["MovieID"]
    liked_indices = movies_reset[movies_reset["MovieID"].isin(liked_ids)].index
    
    taste_vector = embeddings_all[liked_indices].mean(axis=0, keepdims=True)
    sims = cosine_similarity(taste_vector, embeddings_all)[0]
    
    already_rated = profile_df["MovieID"].tolist()
    top_indices = [i for i in sims.argsort()[::-1] if movies_reset.iloc[i]["MovieID"] not in already_rated][:5]
    recommendations = movies_reset.iloc[top_indices][["MovieID", "Title"]]
    
    result = {
        "User_Type": "self",
        "Ratings": my_ratings,
        "Recommendations": recommendations["Title"].tolist()
    }
    
    buffer = io.BytesIO(pd.DataFrame([result]).to_json().encode())
    s3.put_object(Bucket=bucket_name, Key="recommendations/self_user.json", Body=buffer.read())
    
    return result

my_result = create_my_profile(all_movies, all_embeddings, s3, "gomaa-hw2")
print(my_result["Recommendations"])

['Déjà Vu (1997)', 'Afterglow (1997)', 'Bliss (1997)', 'Trial and Error (1997)', 'Firelight (1997)']
